What this notebook does

In this notebook, I estimate customer purchase behavior and 3-month CLV using BG/NBD and Gamma-Gamma, then group customers into value segments.

In [1]:
import pandas as pd 
import numpy as np
import lifetimes

In [2]:
df = pd.read_parquet('cleaned_transactions_full.parquet')

In [3]:
df.shape 

(407664, 9)

In [4]:
df.describe()

,Invoice,Quantity,InvoiceDate,Price,Customer ID,total_amount
count,407664.000000,407664.000000,407664,407664.000000,407664.000000,407664.000000
mean,514761.109151,13.585585,2010-07-01 10:15:11.871688,3.294438,15368.592598,21.664909
min,489434.000000,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,502764.000000,2.000000,2010-03-26 14:01:00,1.250000,13997.000000,4.950000
50%,515304.000000,5.000000,2010-07-09 15:47:00,1.950000,15321.000000,11.900000
75%,527104.000000,12.000000,2010-10-14 17:09:00,3.750000,16812.000000,19.500000
max,538171.000000,19152.000000,2010-12-09 20:01:00,10953.500000,18287.000000,15818.400000
std,14100.789885,96.840747,NaN,34.757965,1679.762138,77.150058


I split the transaction history into a calibration period ending on September 10, 2010 and a holdout period ending on December 9, 2010. The calibration period is used to estimate customer behavior, while the holdout period provides future transactions to compare the predictions against.

In [5]:
from lifetimes.utils import calibration_and_holdout_data

rfm_holdout = calibration_and_holdout_data(
    df,
    customer_id_col="Customer ID",
    datetime_col="InvoiceDate",
    monetary_value_col="total_amount",
    calibration_period_end="2010-09-10",
    observation_period_end="2010-12-09",
    freq="D"
)

rfm_holdout.head()

,frequency_cal,recency_cal,T_cal,monetary_value_cal,frequency_holdout,monetary_value_holdout,duration_holdout
Customer ID,,,,,,,
12346.0,6.0,196.0,270.0,47.143333,0.0,0.000000,90.0
12349.0,1.0,19.0,134.0,200.000000,1.0,25.502182,90.0
12355.0,0.0,0.0,112.0,0.000000,0.0,0.000000,90.0
12358.0,1.0,181.0,276.0,268.100000,1.0,44.394783,90.0
12359.0,4.0,199.0,279.0,312.835000,1.0,17.784839,90.0


I use the BG/NBD model to estimate each customer's purchasing behavior and probability of being still active.

A small penalizer is added to keep the model from overfitting the transaction patterns in the calibration data.

In [6]:
from lifetimes import BetaGeoFitter

bgf = BetaGeoFitter(penalizer_coef=0.001)

bgf.fit(
    frequency=rfm_holdout["frequency_cal"],
    recency=rfm_holdout["recency_cal"],
    T=rfm_holdout["T_cal"]
)

<lifetimes.BetaGeoFitter: fitted with 3373 subjects, a: 0.00, alpha: 66.90, b: 0.19, r: 0.77>

In [7]:
bgf.summary

,coef,se(coef),lower 95% bound,upper 95% bound
r,0.767450,0.029691,0.709255,0.825645
alpha,66.897866,3.043393,60.932815,72.862917
a,0.002804,0.004064,-0.005161,0.010769
b,0.194527,0.249640,-0.294768,0.683822


In [8]:
rfm_holdout["predicted_purchases_90d"] = (
    bgf.conditional_expected_number_of_purchases_up_to_time(
        90,
        rfm_holdout["frequency_cal"],
        rfm_holdout["recency_cal"],
        rfm_holdout["T_cal"]
    )
)

C:\Users\ENVY\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [9]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(
    rfm_holdout["frequency_holdout"],
    rfm_holdout["predicted_purchases_90d"]
)

mae = mean_absolute_error(
    rfm_holdout["frequency_holdout"],
    rfm_holdout["predicted_purchases_90d"]
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

ValueError: Input contains NaN.

compare the predicted number of purchases with the actual purchases in the holdout period using RMSE and MAE.

These metrics show how far the predicted purchase counts are from the actual values, with lower scores indicating better predictions.

I got an Input contains NaN error, so I checked where the missing values were coming from.

In [ ]:
rfm_holdout[["frequency_holdout", "predicted_purchases_90d"]].isna().sum()

frequency_holdout           0
predicted_purchases_90d    87
dtype: int64

There are 87 missing predicted values, while the actual holdout purchase counts have no missing values. This means the NaNs are coming from the BG/NBD predictions rather than the holdout data.

In [ ]:
rfm_holdout[
    rfm_holdout["predicted_purchases_90d"].isna()
].head()

,frequency_cal,recency_cal,T_cal,monetary_value_cal,frequency_holdout,monetary_value_holdout,duration_holdout,predicted_purchases_90d
Customer ID,,,,,,,,
12414.0,0.0,0.0,8.0,0.0,0.0,0.000000,90.0,NaN
12423.0,0.0,0.0,10.0,0.0,1.0,17.195556,90.0,NaN
12460.0,0.0,0.0,1.0,0.0,0.0,0.000000,90.0,NaN
12573.0,0.0,0.0,10.0,0.0,3.0,4.937195,90.0,NaN
12605.0,0.0,0.0,3.0,0.0,3.0,15.741290,90.0,NaN


The NaN predictions are coming from customers who made one purchase during the calibration period. Because they did not purchase again before the cutoff, BG/NBD has no repeat-purchase pattern to estimate for them.

In [ ]:
valid_predictions = rfm_holdout["predicted_purchases_90d"].notna()

print(f"Valid predictions: {valid_predictions.sum()}")
print(f"Missing predictions: {(~valid_predictions).sum()}")

Valid predictions: 3286
Missing predictions: 87


In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(
    rfm_holdout.loc[valid_predictions, "frequency_holdout"],
    rfm_holdout.loc[valid_predictions, "predicted_purchases_90d"]
)

mae = mean_absolute_error(
    rfm_holdout.loc[valid_predictions, "frequency_holdout"],
    rfm_holdout.loc[valid_predictions, "predicted_purchases_90d"]
)

print(f"Customers Evaluated: {valid_predictions.sum()}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

Customers Evaluated: 3286
RMSE: 1.5573
MAE: 0.9597


The model was evaluated on 3,286 customers with valid predictions. The RMSE is 1.5573 and the MAE is 0.9597, meaning the predicted number of purchases is, on average, about 0.96 purchases away from the actual holdout value.

The 87 customers with missing predictions are kept separately and are not included in this evaluation.

In [ ]:
missing_predictions = rfm_holdout[~valid_predictions]

missing_predictions.head()

,frequency_cal,recency_cal,T_cal,monetary_value_cal,frequency_holdout,monetary_value_holdout,duration_holdout,predicted_purchases_90d
Customer ID,,,,,,,,
12414.0,0.0,0.0,8.0,0.0,0.0,0.000000,90.0,NaN
12423.0,0.0,0.0,10.0,0.0,1.0,17.195556,90.0,NaN
12460.0,0.0,0.0,1.0,0.0,0.0,0.000000,90.0,NaN
12573.0,0.0,0.0,10.0,0.0,3.0,4.937195,90.0,NaN
12605.0,0.0,0.0,3.0,0.0,3.0,15.741290,90.0,NaN


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

valid = rfm_holdout["predicted_purchases_90d"].notna()

actual = rfm_holdout.loc[valid, "frequency_holdout"]

actual_mean = actual.mean()

pred_zero = np.zeros(len(actual))

mae_zero = mean_absolute_error(actual, pred_zero)
rmse_zero = root_mean_squared_error(actual, pred_zero)

mean_purchase_rate = (
    rfm_holdout.loc[valid, "frequency_cal"] /
    rfm_holdout.loc[valid, "T_cal"].replace(0, np.nan)
).mean()

pred_mean = np.full(len(actual), mean_purchase_rate * 90)

mae_mean = mean_absolute_error(actual, pred_mean)
rmse_mean = root_mean_squared_error(actual, pred_mean)

model_pred = rfm_holdout.loc[valid, "predicted_purchases_90d"]

print(f"Actual Holdout Mean Purchases: {actual_mean:.4f}")
print("---")
print(f"Baseline (Predict Zero) -> MAE: {mae_zero:.4f} | RMSE: {rmse_zero:.4f}")
print(f"Baseline (Predict Mean) -> MAE: {mae_mean:.4f} | RMSE: {rmse_mean:.4f}")
print(
    f"BG/NBD Model            -> "
    f"MAE: {mean_absolute_error(actual, model_pred):.4f} | "
    f"RMSE: {root_mean_squared_error(actual, model_pred):.4f}"
)

Actual Holdout Mean Purchases: 1.3220
---
Baseline (Predict Zero) -> MAE: 1.3220 | RMSE: 2.5982
Baseline (Predict Mean) -> MAE: 1.1683 | RMSE: 2.2612
BG/NBD Model            -> MAE: 0.9597 | RMSE: 1.5573


BG/NBD performs better than both simple baselines. Its MAE drops from 1.1683 with the mean baseline to 0.9597, while RMSE drops from 2.2612 to 1.5573.

This means that my model is better than simply predicting zero or using the average number of purchases for every customer ;-)

In [ ]:
returning_customers = rfm_holdout[
    rfm_holdout["frequency_cal"] > 0
]

In [ ]:
from lifetimes import GammaGammaFitter

ggf = GammaGammaFitter(penalizer_coef=0.01)

ggf.fit(
    returning_customers["frequency_cal"],
    returning_customers["monetary_value_cal"]
)

ggf.summary

,coef,se(coef),lower 95% bound,upper 95% bound
p,3.775741,0.113811,3.552671,3.998811
q,0.336342,0.008481,0.319720,0.352964
v,3.649665,0.115117,3.424036,3.875294


I use the Gamma-Gamma model to estimate the average monetary value of a customer's future purchases. It is fitted on returning customers because it uses their repeat-purchase history and monetary value.

In [ ]:
returning_customers["expected_avg_spend"] = (
    ggf.conditional_expected_average_profit(
        returning_customers["frequency_cal"],
        returning_customers["monetary_value_cal"]
    )
)

In [ ]:
returning_customers["CLV_3M"] = ggf.customer_lifetime_value(
    bgf,
    returning_customers["frequency_cal"],
    returning_customers["recency_cal"],
    returning_customers["T_cal"],
    returning_customers["monetary_value_cal"],
    time=3,
    discount_rate=0.01,
    freq="D"
)

I combine the BG/NBD purchase predictions with the Gamma-Gamma monetary value estimates to calculate each customer's expected value over the next 3 months.

A 1% discount rate is used to account for the time value of future purchases.

In [ ]:
cols = [
    "frequency_cal",
    "monetary_value_cal",
    "expected_avg_spend",
    "CLV_3M"
]

returning_customers[cols].sort_values(
    by="CLV_3M",
    ascending=False
).head()

,frequency_cal,monetary_value_cal,expected_avg_spend,CLV_3M
Customer ID,,,,
18102.0,35.0,7621.583714,7660.157064,69047.503483
14646.0,29.0,5618.538276,5652.926500,42532.730684
14156.0,53.0,2902.082075,2911.807641,39460.594620
13694.0,41.0,2731.646829,2743.497348,29131.381824
14911.0,79.0,1089.396582,1091.872112,21953.257364


The CLV results look reasonable. The highest-value customers tend to have both frequent purchases and high expected spending, which leads to a much higher estimated 3-month CLV.

In [ ]:
returning_customers["p_alive"] = bgf.conditional_probability_alive(
    returning_customers["frequency_cal"],
    returning_customers["recency_cal"],
    returning_customers["T_cal"]
)

In [ ]:
returning_customers["clv_segment"] = pd.qcut(
    returning_customers["CLV_3M"],
    q=4,
    labels=[
        "D - Low Value",
        "C - Medium",
        "B - High Value",
        "A - VIP / Whale"
    ]
)

segment_analysis = returning_customers.groupby(
    "clv_segment",
    observed=True
).agg(
    customer_count=("CLV_3M", "count"),
    mean_clv=("CLV_3M", "mean"),
    total_clv=("CLV_3M", "sum"),
    mean_p_alive=("p_alive", "mean"),
    mean_frequency=("frequency_cal", "mean")
)

segment_analysis

,customer_count,mean_clv,total_clv,mean_p_alive,mean_frequency
clv_segment,,,,,
D - Low Value,505,114.668090,5.790739e+04,0.963587,1.518812
C - Medium,505,268.194777,1.354384e+05,0.982636,2.134653
B - High Value,504,489.298023,2.466062e+05,0.991187,3.478175
A - VIP / Whale,505,2048.947593,1.034719e+06,0.994203,8.198020


The segments show a clear pattern. As CLV increases from D to A, customers also purchase more frequently and have a higher probability of being active.

In [ ]:
cols_to_export = [
    "frequency_cal",
    "monetary_value_cal",
    "expected_avg_spend",
    "CLV_3M",
    "p_alive",
    "clv_segment"
]

df_clv_export = (
    returning_customers[cols_to_export]
    .reset_index()
)

df_clv_export.to_csv(
    "clv_predictions.csv",
    index=False
)

print("Export complete. File saved as 'clv_predictions.csv'")

Export complete. File saved as 'clv_predictions.csv'
